In [21]:
import os
import torch
import torch.nn.functional as F
from pathlib import Path
from typing import List, Dict
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from plotly.subplots import make_subplots

1. compute norm(W[t+1]- W[t])-norm(W[t]-W[t-1]) for each checkpointing period (try with L2 norm, cosine distance)
2. compute [f[t+1]-f[t]] - [f[t]-f[t-1]]

## 1. compute matrix norm (frobenius and cosine distance)

### Useful Functions

In [15]:
def load_flat_tensor(path):
        return torch.load(path).flatten()

In [16]:
def get_filename(epoch: int, batch: int = None):
        if batch is not None:
            return f"{weight_type}_epoch_{epoch}_batch_{batch}.pt.pt"
        else:
            return f"{weight_type}_epoch_{epoch}.pt.pt"

In [17]:
checkpoints = (
        [(1, i) for i in range(0, 301)] +
        [(1, i) for i in range(400, 9300, 100)] +
        [(i, None) for i in range(1, 41)]
    )

In [26]:
def compute_weight_norm(checkpoint_directory, weight_type, checkpoints):
    
    checkpoint_dir = Path(checkpoint_directory)
    paths = [checkpoint_dir / get_filename(epoch, batch) for epoch, batch in checkpoints]
    paths = [p for p in paths if p.exists()]

    # Sanity check
    if len(paths) < 3:
        raise ValueError("Not enough checkpoints to compute second differences.")

    frob_diffs, cosine_diffs = [], []

    for i in range(1, len(paths) - 1):
        w_prev = load_flat_tensor(paths[i - 1])
        w_curr = load_flat_tensor(paths[i])
        w_next = load_flat_tensor(paths[i + 1])

        #second_diff = w_next - 2 * w_curr + w_prev
        delta_prev = w_curr - w_prev
        delta_next = w_next - w_curr

        frob = torch.norm(delta_next, p=2).item()-torch.norm(delta_prev, p=2).item()
        cosine = F.cosine_similarity(delta_prev, delta_next, dim=0).item()

        frob_diffs.append(frob)
        cosine_diffs.append(cosine)

    return {
        "frob": frob_diffs,
        "cosine": cosine_diffs
    }

In [19]:
def plot_breakthroughs(metrics: dict, checkpoints: list, title: str = "Breakthrough Analysis"):
    # Skip the first and last elements due to diff computation
    x_labels = checkpoints[1:-1]  # Format: (epoch, batch)

    # Format nicely for readability on the x-axis
    x_ticks = [f"e{e}_b{b}" for (e, b) in x_labels]

    fig = go.Figure()

    # Frobenius norm second difference
    fig.add_trace(go.Scatter(
        x=x_ticks,
        y=metrics["frob"],
        mode='lines+markers',
        name='Frobenius Norm',
        line=dict(color='royalblue')
    ))

    # Cosine distance second difference
    fig.add_trace(go.Scatter(
        x=x_ticks,
        y=metrics["cosine"],
        mode='lines+markers',
        name='Cosine Distance',
        line=dict(color='firebrick')
    ))

    fig.update_layout(
        title=title,
        xaxis_title='Checkpoint (Epoch_Batch)',
        yaxis_title='Breakthrough Magnitude',
        xaxis=dict(tickangle=45, tickmode='array', tickvals=x_ticks[::10], ticktext=x_ticks[::10]),
        legend=dict(
            x=1.02,
            y=1,
            traceorder="normal",
            bordercolor="Black",
            borderwidth=1,
            xanchor="left"
        ),
        template='plotly_white',
        width=1000,
        height=500
    )

    fig.show()


### Actual Plots

In [34]:
def add_breakthrough_subplot(fig, row, col, metrics, checkpoints, title):
    x_labels = checkpoints[1:-1]
    x_ticks = [f"e{e}_b{b}" if b is not None else f"e{e}" for (e, b) in x_labels]

    fig.add_trace(go.Scatter(
        x=x_ticks,
        y=metrics["frob"],
        mode='lines+markers',
        name='Frobenius Norm',
        line=dict(color='royalblue'),
        showlegend=(row == 1)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=x_ticks,
        y=metrics["cosine"],
        mode='lines+markers',
        name='Cosine Distance',
        line=dict(color='firebrick'),
        showlegend=(row == 1)
    ), row=row, col=col)

    fig.update_xaxes(
        title_text="Checkpoint (Epoch_Batch)",
        row=row,
        col=col,
        tickangle=45,
        tickmode='array',
        tickvals=x_ticks[::max(1, len(x_ticks) // 10)],
        ticktext=x_ticks[::max(1, len(x_ticks) // 10)]
    )

    fig.update_yaxes(
        title_text="Breakthrough Magnitude",
        row=row,
        col=col
    )




In [35]:
def plot_all_breakthroughs(checkpoint_directory, weight_type, plot_title, html_save):
    fig = make_subplots(
        rows=3,
        cols=1,
        subplot_titles=[
            '300 First Batches',
            'Remaining of the First Epoch',
            '40 Epochs'
        ]
    )

    # 1. First 300 batches
    checkpoints1 = [(1, i) for i in range(0, 301)]
    metrics1 = compute_weight_norm(checkpoint_directory, weight_type, checkpoints1)
    add_breakthrough_subplot(fig, 1, 1, metrics1, checkpoints1, '300 First Batches')

    # 2. Remaining of the first epoch
    checkpoints2 = [(1, i) for i in range(400, 9300, 100)]
    metrics2 = compute_weight_norm(checkpoint_directory, weight_type, checkpoints2)
    add_breakthrough_subplot(fig, 2, 1, metrics2, checkpoints2, 'Remaining of the First Epoch')

    # 3. Full 40 epochs
    checkpoints3 = [(i, None) for i in range(1, 41)]
    metrics3 = compute_weight_norm(checkpoint_directory, weight_type, checkpoints3)
    add_breakthrough_subplot(fig, 3, 1, metrics3, checkpoints3, '40 Epochs')

    fig.update_layout(
        height=1200,
        width=1000,
        title_text=f"Breakthrough Analysis of the Recurrent Weight of {plot_title}",
        template='plotly_white'
    )

    fig.show()
    fig.write_html(f'/scratch2/mrenaudin/colorlessgreenRNNs/docs/plots/{html_save}.html')


In [ ]:
weight_type = 'layer1_input_gate_hh'
checkpoint_directory = '/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam_full_check_shuffled/weights'

plot_all_breakthroughs(checkpoint_directory, weight_type,'the Input Gate of the 1st Layer', 'l1_input_hh' )

weight_type = 'layer1_output_gate_hh'
checkpoint_directory = '/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam_full_check_shuffled/weights'

plot_all_breakthroughs(checkpoint_directory, weight_type,'the Output Gate of the 1st Layer', 'l1_output_hh' )

In [ ]:
weight_type = 'layer1_forget_gate_hh'
checkpoint_directory = '/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam_full_check_shuffled/weights'

plot_all_breakthroughs(checkpoint_directory, weight_type,'the Forget Gate of the 1st Layer', 'l1_forget_hh' )

## 2. Compute breakthrough on Nounpp conditions

In [52]:
df = pd.read_csv('/scratch2/mrenaudin/colorlessgreenRNNs/evaluation_notebooks/results/lstm_adam_full_check_shuffled')

In [ ]:
df

In [57]:
checkpoints = (
        [(1, i) for i in range(0, 301)] +
        [(1, i) for i in range(400, 9300, 100)] +
        [(i, None) for i in range(1, 41)]
    )

In [58]:
df['checkpoints']= checkpoints

In [59]:
df_cool = df

In [61]:
def get_samples(df,valid_checkpoints):

    #300 first batches
    #filter df
    df_filtered = df[df['checkpoints'].isin(valid_checkpoints)].copy().reset_index(drop=True)

    # # Sort by checkpoint second element to ensure order
    # df_filtered['checkpoint_idx'] = df_filtered['checkpoints'].apply(lambda x: x[1])
    # df_filtered = df_filtered.sort_values('checkpoint_idx').reset_index(drop=True)

    # Conditions to calculate for
    conditions = ['singular singular', 'singular plural', 'plural singular', 'plural plural']

    # Initialize dictionary to store results
    second_diff_results = {cond: [] for cond in conditions}

    # Compute second difference for indices 1 to len-2 (since we need t-1 and t+1)
    for i in range(1, len(df_filtered) - 1):
        for cond in conditions:
            f_t_plus_1 = df_filtered.loc[i+1, cond]
            f_t = df_filtered.loc[i, cond]
            f_t_minus_1 = df_filtered.loc[i-1, cond]
            second_diff = f_t_plus_1 - 2*f_t + f_t_minus_1
            second_diff_results[cond].append(second_diff)

    # For example, create a new DataFrame for second differences (length 299 since edges are excluded)
    second_diff_df = pd.DataFrame(second_diff_results)
    #second_diff_df['checkpoint_idx'] = df_filtered.loc[1:-1, 'checkpoint_idx'].values

    print(second_diff_df)
    return second_diff_df

    
    

In [87]:
def add_subplot(fig, checkpoints, sample, conditions, row, col, show_legend):
    x_labels = checkpoints[1:-1]
    x_ticks = [f"e{e}_b{b}" if b is not None else f"e{e}" for (e, b) in x_labels]
    line_styles = {
        'singular singular': dict(color='blue', dash='solid'),
        'singular plural': dict(color='blue', dash='dash'),
        'plural singular': dict(color='red', dash='dash'),
        'plural plural': dict(color='red', dash='solid'),
    }
    for cond in conditions:
        fig.add_trace(go.Scatter(
            x=x_ticks,
            y=sample[cond],
            mode='lines+markers',
            name=cond,
            line = line_styles[cond],
            showlegend= show_legend
        ), row=row, col=col) 


    fig.update_xaxes(
        title_text="Checkpoint (Epoch_Batch)",
        row=row,
        col=col,
        tickangle=45,
        tickmode='array',
        tickvals=x_ticks[::max(1, len(x_ticks) // 10)],
        ticktext=x_ticks[::max(1, len(x_ticks) // 10)]
    )

    fig.update_yaxes(
        title_text="Breakthrough Magnitude",
        row=row,
        col=col
    )




In [88]:
def plot_all_break(df, conditions, plot_title, html_save):
    fig = make_subplots(
        rows=3,
        cols=1,
        subplot_titles=[
            '300 First Batches',
            'Remaining of the First Epoch',
            '40 Epochs'
        ]
    )
    checkpoints1 = [(1, i) for i in range(0, 301)]
    sample1 = get_samples(df,checkpoints1)
    add_subplot(fig, checkpoints1,sample1, conditions, 1, 1, show_legend=True)
    # 2. Remaining of the first epoch
    checkpoints2 = [(1, i) for i in range(400, 9300, 100)]
    sample2 = get_samples(df, checkpoints2)    
    add_subplot(fig, checkpoints2,sample2,  conditions, 2, 1, show_legend=False)
    # 3. Full 40 epochs
    checkpoints3 = [(i, None) for i in range(1, 41)]
    sample3 = get_samples(df, checkpoints3)
    print('sample3',sample3)
    add_subplot(fig, checkpoints3,sample3,  conditions, 3, 1, show_legend=False)
    
    fig.update_layout(
        height=1200,
        width=1000,
        title_text=f"Breakthrough Analysis of the Recurrent Weight of {plot_title}",
        template='plotly_white'
    )

    fig.show()
    fig.write_html(f'/scratch2/mrenaudin/colorlessgreenRNNs/docs/plots/{html_save}.html')

In [ ]:
df_cool

In [ ]:
conditions = ['singular singular', 'singular plural', 'plural singular', 'plural plural']
plot_all_break(df_cool, conditions, 'NounPP', 'Nounpp')